# SFpark Parking Dataset — Feature Engineering Pipeline v2.2 (3-hour lookback)

This notebook prepares leakage-safe, contiguous hourly features for next-hour GMP parking-availability forecasting.

### Main corrections

1. Temporal gaps are enforced: lags, changes, rolling statistics, and targets never cross non-1-hour gaps.
2. GMP validity is enforced: missing/non-operating GMP occupancy is not imputed.
3. The supervised table is genuinely model-ready: inputs, history, and next-hour target must all be valid.
4. Future leakage is avoided: no full-dataset future maximum is used for capacity/coverage features.
5. The lookback is fixed at **3 hours**, matching the current modeling plan.
6. Features requiring more than 3 historical observations are not included in the primary model-input set, so the 3-hour requirement is real rather than nominal.
7. The primary target is next-hour GMP availability (`1 - GMP occupancy`). Next-hour occupancy is retained as an alternate target.

The notebook stops before model training.


## 1. Data interpretation and feature policy

One row represents one `BLOCK_ID` at one hourly timestamp. No 5-minute observations are manufactured.

The core occupancy definition is:

\[
GMP\ Occupancy_t = 
rac{GMP\ Occupied\ Time}{GMP\ Occupied\ Time + GMP\ Vacant\ Time}
\]

and:

\[
GMP\ Availability_t = 1-GMP\ Occupancy_t
\]

Unknown time is excluded from the occupancy denominator. Missing GMP occupancy therefore means **unusable for a GMP forecasting sample**, not zero occupancy.

In [1]:
# 2. Imports and configuration
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# EDIT THIS PATH ON YOUR SYSTEM.
# The pipeline expects the original SFpark hourly analytical CSV.
# ------------------------------------------------------------------
DATA_PATH = Path(
    r"C:/Users/Pulkit Gupta/OneDrive/Desktop/SC/SC_PROJECT/SFpark_ParkingSensorData_HourlyOccupancy_20112013.csv"
)

# Quality / sequence configuration
MAX_UNKNOWN_RATE = 0.50
LOOKBACK_HOURS = 3
HORIZON_HOURS = 1

# Dataset source period / documented outage
DATA_START = pd.Timestamp("2011-04-01 00:00:00")
OUTAGE_START = pd.Timestamp("2012-12-10 00:00:00")
OUTAGE_END = pd.Timestamp("2012-12-17 23:59:59")
EXCLUDE_WEST_PORTAL = True

# Whether a model-ready row must have all selected input features non-null.
REQUIRE_COMPLETE_INPUTS = True

print("Configuration loaded.")
print("LOOKBACK_HOURS:", LOOKBACK_HOURS)
print("HORIZON_HOURS:", HORIZON_HOURS)


Configuration loaded.
LOOKBACK_HOURS: 3
HORIZON_HOURS: 1


In [2]:
# 3. Load raw SFpark hourly dataset
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. Change DATA_PATH to the location of the original SFpark CSV."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)
print("Original shape:", df.shape)
print("Date column present:", "START_TIME_DT" in df.columns)

Original shape: (1048575, 33)
Date column present: True


In [3]:
# 4. Validate required source columns
REQUIRED_COLUMNS = [
    "BLOCK_ID", "STREET_NAME", "BLOCK_NUM", "STREET_BLOCK",
    "AREA_TYPE", "PM_DISTRICT_NAME", "RATE",
    "START_TIME_DT", "TOTAL_TIME",
    "TOTAL_OCCUPIED_TIME", "TOTAL_VACANT_TIME", "TOTAL_UNKNOWN_TIME",
    "OP_TIME", "OP_OCCUPIED_TIME", "OP_VACANT_TIME", "OP_UNKNOWN_TIME",
    "NONOP_TIME", "NONOP_OCCUPIED_TIME", "NONOP_VACANT_TIME", "NONOP_UNKNOWN_TIME",
    "GMP_TIME", "GMP_OCCUPIED_TIME", "GMP_VACANT_TIME", "GMP_UNKNOWN_TIME",
    "COMM_TIME", "COMM_OCCUPIED_TIME", "COMM_VACANT_TIME", "COMM_UNKNOWN_TIME"
]
missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing:
    raise ValueError("Missing required source columns:\n" + "\n".join(missing))
print("All required columns are present.")

All required columns are present.


In [4]:
# 5. Data types and timestamp parsing
DF_NUMERIC = [
    "BLOCK_ID", "BLOCK_NUM", "RATE",
    "TOTAL_TIME", "TOTAL_OCCUPIED_TIME", "TOTAL_VACANT_TIME", "TOTAL_UNKNOWN_TIME",
    "OP_TIME", "OP_OCCUPIED_TIME", "OP_VACANT_TIME", "OP_UNKNOWN_TIME",
    "NONOP_TIME", "NONOP_OCCUPIED_TIME", "NONOP_VACANT_TIME", "NONOP_UNKNOWN_TIME",
    "GMP_TIME", "GMP_OCCUPIED_TIME", "GMP_VACANT_TIME", "GMP_UNKNOWN_TIME",
    "COMM_TIME", "COMM_OCCUPIED_TIME", "COMM_VACANT_TIME", "COMM_UNKNOWN_TIME"
]

df["START_TIME_DT"] = pd.to_datetime(df["START_TIME_DT"], errors="coerce")
for col in DF_NUMERIC:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["BLOCK_ID", "START_TIME_DT"]).copy()
print("Rows after type cleaning:", len(df))
print("Date range:", df["START_TIME_DT"].min(), "→", df["START_TIME_DT"].max())

Rows after type cleaning: 482769
Date range: 2011-01-04 00:00:00 → 2013-12-05 23:00:00


In [5]:
# 6. Source-specific exclusions
before = len(df)

# Do not train on pre-April-2011 observations used outside the documented analytical period.
df = df.loc[df["START_TIME_DT"] >= DATA_START].copy()

# Exclude West Portal, which the source documentation treats as unreliable.
if EXCLUDE_WEST_PORTAL:
    df = df.loc[
        df["PM_DISTRICT_NAME"].astype(str).str.strip().str.lower().ne("west portal")
    ].copy()

# Remove the documented unrecoverable outage week.
df = df.loc[~df["START_TIME_DT"].between(OUTAGE_START, OUTAGE_END)].copy()

print("Rows removed:", before - len(df))
print("Rows remaining:", len(df))

Rows removed: 21410
Rows remaining: 461359


In [6]:
# 7. Sort at the fundamental grain and reject duplicate block-hours

df = df.sort_values(["BLOCK_ID", "START_TIME_DT"]).reset_index(drop=True)

dup_mask = df.duplicated(["BLOCK_ID", "START_TIME_DT"], keep=False)
print("Duplicate block-hour rows:", int(dup_mask.sum()))
if dup_mask.any():
    display(df.loc[dup_mask, ["BLOCK_ID", "START_TIME_DT"]].head(25))
    raise ValueError("Duplicate BLOCK_ID + START_TIME_DT observations exist; resolve them before feature engineering.")

Duplicate block-hour rows: 0


In [7]:
# 8. Safe ratio helper

def safe_ratio(numerator, denominator):
    denominator = denominator.replace(0, np.nan)
    return numerator / denominator

In [8]:
# 9. Occupancy and availability features

df["TOTAL_OCCUPANCY"] = safe_ratio(
    df["TOTAL_OCCUPIED_TIME"],
    df["TOTAL_OCCUPIED_TIME"] + df["TOTAL_VACANT_TIME"]
)
df["TOTAL_AVAILABILITY"] = 1 - df["TOTAL_OCCUPANCY"]

df["OP_OCCUPANCY"] = safe_ratio(
    df["OP_OCCUPIED_TIME"],
    df["OP_OCCUPIED_TIME"] + df["OP_VACANT_TIME"]
)
df["OP_AVAILABILITY"] = 1 - df["OP_OCCUPANCY"]

df["GMP_OCCUPANCY"] = safe_ratio(
    df["GMP_OCCUPIED_TIME"],
    df["GMP_OCCUPIED_TIME"] + df["GMP_VACANT_TIME"]
)
df["GMP_AVAILABILITY"] = 1 - df["GMP_OCCUPANCY"]

df["COMM_OCCUPANCY"] = safe_ratio(
    df["COMM_OCCUPIED_TIME"],
    df["COMM_OCCUPIED_TIME"] + df["COMM_VACANT_TIME"]
)

In [9]:
# 10. Quality and monitored-space features

df["TOTAL_UNKNOWN_RATE"] = safe_ratio(df["TOTAL_UNKNOWN_TIME"], df["TOTAL_TIME"])
df["OP_UNKNOWN_RATE"] = safe_ratio(df["OP_UNKNOWN_TIME"], df["OP_TIME"])
df["GMP_UNKNOWN_RATE"] = safe_ratio(df["GMP_UNKNOWN_TIME"], df["GMP_TIME"])

# TOTAL_TIME is the aggregate number of sensor-seconds represented during the hour.
df["EFFECTIVE_MONITORED_SPACES"] = df["TOTAL_TIME"] / 3600.0

# Do NOT calculate COVERAGE_RATIO using a full-dataset block maximum.
# Such a maximum can contain future information. The primary pipeline omits it.

# Current-row quality. The total-time screen is retained for source-level quality;
# GMP quality is separately enforced for GMP forecasting samples.
df["QUALITY_OK"] = (
    df["TOTAL_UNKNOWN_RATE"].notna()
    & df["TOTAL_UNKNOWN_RATE"].le(MAX_UNKNOWN_RATE)
)

df["GMP_QUALITY_OK"] = (
    df["GMP_UNKNOWN_RATE"].notna()
    & df["GMP_UNKNOWN_RATE"].le(MAX_UNKNOWN_RATE)
    & df["GMP_OCCUPANCY"].notna()
)

In [10]:
# 11. Operating / restriction context

df["NONOP_RATE"] = safe_ratio(df["NONOP_TIME"], df["TOTAL_TIME"])
df["GMP_COMM_SHARE"] = safe_ratio(df["COMM_TIME"], df["OP_TIME"])

In [11]:
# 12. Time features

df["HOUR"] = df["START_TIME_DT"].dt.hour
df["DAY_OF_WEEK"] = df["START_TIME_DT"].dt.dayofweek
df["WEEKEND"] = df["DAY_OF_WEEK"].isin([5, 6]).astype(int)
df["MONTH"] = df["START_TIME_DT"].dt.month
df["DAY_OF_YEAR"] = df["START_TIME_DT"].dt.dayofyear

# Cyclic encodings avoid artificial discontinuities (e.g. 23:00 vs 00:00).
df["HOUR_SIN"] = np.sin(2 * np.pi * df["HOUR"] / 24.0)
df["HOUR_COS"] = np.cos(2 * np.pi * df["HOUR"] / 24.0)
df["DOW_SIN"] = np.sin(2 * np.pi * df["DAY_OF_WEEK"] / 7.0)
df["DOW_COS"] = np.cos(2 * np.pi * df["DAY_OF_WEEK"] / 7.0)

## 13. Build genuine contiguous time-series segments

This is the key correction relative to the previous pipeline.

A `shift(1)` is only a true one-hour lag when the previous observation is exactly one hour earlier. Therefore the notebook creates a new `SEQUENCE_ID` whenever the timestamp gap within a block is not exactly one hour.

GMP-invalid observations also break a **GMP-valid sequence**, because a forecasting history must not contain unknown GMP occupancy values.

In [12]:
# 14. Sequence continuity markers

df["TIME_DIFF_HOURS"] = (
    df.groupby("BLOCK_ID")["START_TIME_DT"].diff().dt.total_seconds() / 3600.0
)

df["SEQUENCE_BREAK"] = (
    df["TIME_DIFF_HOURS"].notna()
    & ~np.isclose(df["TIME_DIFF_HOURS"], 1.0)
)

# Pure chronological segment ID: increments after every temporal gap.
df["SEQUENCE_ID"] = (
    df.groupby("BLOCK_ID")["SEQUENCE_BREAK"].cumsum().astype("int64")
)

# GMP-valid segment ID: breaks on either a temporal gap or unusable GMP occupancy.
df["GMP_VALID_BREAK"] = df["SEQUENCE_BREAK"] | ~df["GMP_QUALITY_OK"]
df["GMP_SEQUENCE_ID"] = (
    df.groupby("BLOCK_ID")["GMP_VALID_BREAK"].cumsum().astype("int64")
)

gap_count = int(df["SEQUENCE_BREAK"].sum())
print("Rows following a non-1-hour gap:", gap_count)

Rows following a non-1-hour gap: 68347


In [13]:
# 15. Correct block + sequence grouping for temporal features
# Every lag/rolling calculation is guaranteed to stay inside a contiguous segment.
# The primary model inputs are intentionally limited to features compatible with a
# 3-hour history. Six-hour rolling features are omitted so they cannot silently
# force the effective lookback back to 6 hours.

g_seq = df.groupby(["BLOCK_ID", "SEQUENCE_ID"], sort=False, group_keys=False)

# GMP lags
df["GMP_OCC_LAG_1"] = g_seq["GMP_OCCUPANCY"].shift(1)
df["GMP_OCC_LAG_2"] = g_seq["GMP_OCCUPANCY"].shift(2)
df["GMP_OCC_LAG_3"] = g_seq["GMP_OCCUPANCY"].shift(3)

df["GMP_OCC_CHANGE_1"] = df["GMP_OCCUPANCY"] - df["GMP_OCC_LAG_1"]
df["GMP_OCC_CHANGE_3"] = df["GMP_OCCUPANCY"] - df["GMP_OCC_LAG_3"]

df["RATE_CHANGE"] = g_seq["RATE"].diff()

# Only a 3-hour rolling statistic is used in the primary model.
df["GMP_ROLLING_MEAN_3"] = g_seq["GMP_OCCUPANCY"].transform(
    lambda s: s.rolling(3, min_periods=3).mean()
)


## 16. No >3-hour rolling features in the primary pipeline

The primary pipeline uses a **3-hour model lookback**. Therefore rolling statistics
that require 6 observations (for example a 6-hour standard deviation or slope) are
not included in the model-input table. This keeps the configured lookback honest and
prevents a hidden 6-hour minimum history requirement.


## 17. Next-hour target construction

The target is created only from the next contiguous row in the same block.

Primary target:

`TARGET_GMP_AVAILABILITY_1H = GMP_AVAILABILITY[t+1]`

Alternate target:

`TARGET_GMP_OCCUPANCY_1H = GMP_OCCUPANCY[t+1]`

In [14]:
# 18. Next-hour target and target-quality checks

future_time = df.groupby("BLOCK_ID")["START_TIME_DT"].shift(-HORIZON_HOURS)
df["TARGET_TIME_DIFF_HOURS"] = (
    future_time - df["START_TIME_DT"]
).dt.total_seconds() / 3600.0

df["TARGET_CONTIGUOUS"] = np.isclose(
    df["TARGET_TIME_DIFF_HOURS"], HORIZON_HOURS
)

# Shift within block is safe because target contiguity is explicitly checked.
df["TARGET_GMP_OCCUPANCY_1H"] = (
    df.groupby("BLOCK_ID")["GMP_OCCUPANCY"].shift(-HORIZON_HOURS)
)
df["TARGET_GMP_AVAILABILITY_1H"] = 1 - df["TARGET_GMP_OCCUPANCY_1H"]

df["TARGET_GMP_UNKNOWN_RATE_1H"] = (
    df.groupby("BLOCK_ID")["GMP_UNKNOWN_RATE"].shift(-HORIZON_HOURS)
)

df["TARGET_GMP_QUALITY_OK"] = (
    df["TARGET_GMP_OCCUPANCY_1H"].notna()
    & df["TARGET_GMP_UNKNOWN_RATE_1H"].notna()
    & df["TARGET_GMP_UNKNOWN_RATE_1H"].le(MAX_UNKNOWN_RATE)
)

## 19. Enforce the requested historical lookback

`LOOKBACK_HOURS` is a real constraint. The pipeline separately tracks:

- `GMP_HISTORY_LENGTH`: consecutive valid GMP observations ending at the current row.
- `INPUT_HISTORY_LENGTH`: consecutive rows where every selected model input is available, with no temporal gap and no GMP-quality failure.

A row can become model-ready only if the full selected input history is available, the next hour is contiguous, and the next-hour GMP target passes the quality rule.


In [15]:
# 20. Enforce a full contiguous 3-hour history for the actual model inputs
# A model-ready row must have exactly the requested contiguous history available.

PRELIM_INPUT_FEATURES = [
    "GMP_OCCUPANCY", "GMP_AVAILABILITY", "OP_OCCUPANCY", "OP_AVAILABILITY",
    "TOTAL_OCCUPANCY", "TOTAL_UNKNOWN_RATE", "OP_UNKNOWN_RATE",
    "GMP_UNKNOWN_RATE", "EFFECTIVE_MONITORED_SPACES", "NONOP_RATE",
    "GMP_COMM_SHARE", "RATE", "RATE_CHANGE", "HOUR_SIN", "HOUR_COS",
    "DOW_SIN", "DOW_COS", "WEEKEND", "GMP_OCC_LAG_1", "GMP_OCC_LAG_2",
    "GMP_OCC_LAG_3", "GMP_OCC_CHANGE_1", "GMP_OCC_CHANGE_3",
    "GMP_ROLLING_MEAN_3"
]

INPUT_COMPLETE = df[PRELIM_INPUT_FEATURES].notna().all(axis=1)
df["MODEL_INPUT_COMPLETE"] = INPUT_COMPLETE

# Any temporal gap, GMP-quality failure, or missing selected input starts a new
# valid-input run. This guarantees that the 3-hour history does not skip invalid rows.
df["INPUT_SEQUENCE_BREAK"] = (
    df["SEQUENCE_BREAK"] | ~df["GMP_QUALITY_OK"] | ~df["MODEL_INPUT_COMPLETE"]
)
df["INPUT_SEQUENCE_ID"] = (
    df.groupby("BLOCK_ID")["INPUT_SEQUENCE_BREAK"].cumsum().astype("int64")
)

g_input = df.groupby(["BLOCK_ID", "INPUT_SEQUENCE_ID"], sort=False, group_keys=False)
df["INPUT_HISTORY_LENGTH"] = g_input.cumcount() + 1

df["HISTORY_CONTIGUOUS"] = (
    df["MODEL_INPUT_COMPLETE"]
    & df["GMP_QUALITY_OK"]
    & df["INPUT_HISTORY_LENGTH"].ge(LOOKBACK_HOURS)
)

# Consecutive valid GMP-history length, retained for diagnostics.
g_gmp = df.groupby(["BLOCK_ID", "GMP_SEQUENCE_ID"], sort=False, group_keys=False)
df["GMP_HISTORY_LENGTH"] = g_gmp.cumcount() + 1

print("Rows with full contiguous model-input history:", int(df["HISTORY_CONTIGUOUS"].sum()))


Rows with full contiguous model-input history: 63219


In [16]:
# 21. Feature lists

IDENTIFIER_COLUMNS = [
    "BLOCK_ID", "STREET_NAME", "BLOCK_NUM", "STREET_BLOCK",
    "AREA_TYPE", "PM_DISTRICT_NAME", "START_TIME_DT"
]

MODEL_INPUT_FEATURES = [
    # Current parking state
    "GMP_OCCUPANCY",
    "GMP_AVAILABILITY",
    "OP_OCCUPANCY",
    "OP_AVAILABILITY",
    "TOTAL_OCCUPANCY",

    # Quality / supply context
    "TOTAL_UNKNOWN_RATE",
    "OP_UNKNOWN_RATE",
    "GMP_UNKNOWN_RATE",
    "EFFECTIVE_MONITORED_SPACES",

    # Operating context
    "NONOP_RATE",
    "GMP_COMM_SHARE",

    # Price context
    "RATE",
    "RATE_CHANGE",

    # Calendar
    "HOUR_SIN",
    "HOUR_COS",
    "DOW_SIN",
    "DOW_COS",
    "WEEKEND",

    # Temporal dynamics compatible with a 3-hour history
    "GMP_OCC_LAG_1",
    "GMP_OCC_LAG_2",
    "GMP_OCC_LAG_3",
    "GMP_OCC_CHANGE_1",
    "GMP_OCC_CHANGE_3",
    "GMP_ROLLING_MEAN_3",
]

TARGET_COLUMNS = [
    "TARGET_GMP_AVAILABILITY_1H",
    "TARGET_GMP_OCCUPANCY_1H",
]

QUALITY_COLUMNS = [
    "QUALITY_OK",
    "GMP_QUALITY_OK",
    "SEQUENCE_BREAK",
    "SEQUENCE_ID",
    "GMP_SEQUENCE_ID",
    "TIME_DIFF_HOURS",
    "TARGET_TIME_DIFF_HOURS",
    "TARGET_CONTIGUOUS",
    "TARGET_GMP_QUALITY_OK",
    "GMP_HISTORY_LENGTH",
    "MODEL_INPUT_COMPLETE",
    "INPUT_SEQUENCE_BREAK",
    "INPUT_SEQUENCE_ID",
    "INPUT_HISTORY_LENGTH",
    "HISTORY_CONTIGUOUS",
]

FINAL_COLUMNS = list(dict.fromkeys(
    IDENTIFIER_COLUMNS + MODEL_INPUT_FEATURES + QUALITY_COLUMNS + TARGET_COLUMNS
))

print("Input feature count:", len(MODEL_INPUT_FEATURES))
print("Target columns:", TARGET_COLUMNS)


Input feature count: 24
Target columns: ['TARGET_GMP_AVAILABILITY_1H', 'TARGET_GMP_OCCUPANCY_1H']


In [17]:
# 22. Build complete engineered feature table

# GMP_HISTORY_LENGTH is created in the lookback cell before FINAL_COLUMNS is assembled.
missing_final = [c for c in FINAL_COLUMNS if c not in df.columns]
if missing_final:
    raise KeyError(f"FINAL_COLUMNS contains columns not created by the pipeline: {missing_final}")

feature_df = df[FINAL_COLUMNS].copy()
feature_df = feature_df.sort_values(["BLOCK_ID", "START_TIME_DT"]).reset_index(drop=True)

print("Complete engineered shape:", feature_df.shape)


Complete engineered shape: (461359, 48)


In [18]:
# 23. Build the genuinely model-ready table

input_complete = feature_df[MODEL_INPUT_FEATURES].notna().all(axis=1)

MODEL_READY_MASK = (
    feature_df["QUALITY_OK"]
    & feature_df["GMP_QUALITY_OK"]
    & feature_df["HISTORY_CONTIGUOUS"]
    & feature_df["MODEL_INPUT_COMPLETE"]
    & feature_df["TARGET_CONTIGUOUS"]
    & feature_df["TARGET_GMP_QUALITY_OK"]
)

if REQUIRE_COMPLETE_INPUTS:
    MODEL_READY_MASK &= input_complete

model_ready_df = feature_df.loc[MODEL_READY_MASK].copy()

print("All engineered rows:", len(feature_df))
print("Model-ready rows:", len(model_ready_df))
print("Model-ready share:", round(len(model_ready_df) / len(feature_df), 4))
print("Rows rejected for incomplete input features:", int((~input_complete).sum()))

All engineered rows: 461359
Model-ready rows: 50950
Model-ready share: 0.1104
Rows rejected for incomplete input features: 382279


In [22]:
# 24. Critical validation checks

# A. Model-ready rows must have no missing primary inputs.
if REQUIRE_COMPLETE_INPUTS:
    assert model_ready_df[MODEL_INPUT_FEATURES].notna().all().all(),         "Model-ready table contains missing model inputs."

# B. Model-ready targets must exist.
assert model_ready_df[TARGET_COLUMNS].notna().all().all(),     "Model-ready table contains missing targets."

# C. Model-ready target must be exactly one hour ahead.
assert np.isclose(
    model_ready_df["TARGET_TIME_DIFF_HOURS"], HORIZON_HOURS
).all(), "Target horizon is not consistently 1 hour."

# D. Full lookback must actually be present.
assert (model_ready_df["INPUT_HISTORY_LENGTH"] >= LOOKBACK_HOURS).all(), \
    "A model-ready row does not have the requested lookback length."

# D. Sequence breaks / input failures must not occur at model-ready current rows.
assert (~model_ready_df["SEQUENCE_BREAK"]).all(),     "A model-ready current row follows a temporal gap."
assert (~model_ready_df["INPUT_SEQUENCE_BREAK"]).all(),     "A model-ready current row does not belong to a continuous input-feature run."

print("All model-ready validation checks passed.")

All model-ready validation checks passed.


In [20]:
# 25. Verify lags cannot cross temporal gaps

gap_rows = df["SEQUENCE_BREAK"]
non_null_lag_after_gap = df.loc[gap_rows, "GMP_OCC_LAG_1"].notna().sum()
print("Non-null lag-1 values immediately after temporal gaps:", int(non_null_lag_after_gap))
assert non_null_lag_after_gap == 0, "Lag-1 still crosses a temporal gap."

Non-null lag-1 values immediately after temporal gaps: 0


In [21]:
# 26. Verify block boundaries
block_start = df["BLOCK_ID"].ne(df["BLOCK_ID"].shift(1))
assert df.loc[block_start, "GMP_OCC_LAG_1"].isna().all(),     "A lag crossed a BLOCK_ID boundary."
print("Block-boundary lag check passed.")

Block-boundary lag check passed.


## 27. Lookback diagnostics

The primary model now uses a **3-hour** history. The diagnostic below reports how many
contiguous GMP-valid runs are long enough to support 3, 6, and 12-hour experiments.
Only the 3-hour configuration is used by this notebook.


In [23]:
# 28. Contiguous GMP run-length diagnostics
run_lengths = (
    df.loc[df["GMP_QUALITY_OK"]]
      .groupby(["BLOCK_ID", "GMP_SEQUENCE_ID"], sort=False)
      .size()
)

print(run_lengths.describe())
print("\nShare of valid GMP runs >= 3h:", round((run_lengths >= 3).mean(), 4))
print("Share of valid GMP runs >= 6h:", round((run_lengths >= 6).mean(), 4))
print("Share of valid GMP runs >= 12h:", round((run_lengths >= 12).mean(), 4))


count    32272.000000
mean         4.802894
std          4.158585
min          1.000000
25%          1.000000
50%          3.000000
75%          9.000000
max         16.000000
dtype: float64

Share of valid GMP runs >= 3h: 0.5123
Share of valid GMP runs >= 6h: 0.3847
Share of valid GMP runs >= 12h: 0.061


In [24]:
# 29. Dataset summary
summary = pd.Series({
    "all_engineered_rows": len(feature_df),
    "model_ready_rows": len(model_ready_df),
    "model_ready_share": len(model_ready_df) / max(len(feature_df), 1),
    "model_ready_blocks": model_ready_df["BLOCK_ID"].nunique(),
    "temporal_gap_rows": int(feature_df["SEQUENCE_BREAK"].sum()),
    "gmp_quality_rows": int(feature_df["GMP_QUALITY_OK"].sum()),
    "full_3h_lookback_rows": int(feature_df["HISTORY_CONTIGUOUS"].sum()),
    "valid_next_hour_target_rows": int((feature_df["TARGET_CONTIGUOUS"] & feature_df["TARGET_GMP_QUALITY_OK"]).sum()),
})
display(summary.to_frame("value"))

block_counts = model_ready_df.groupby("BLOCK_ID").size()
print("\nModel-ready samples per block:")
display(block_counts.describe().to_frame("samples_per_block"))


,value
all_engineered_rows,461359.000000
model_ready_rows,50950.000000
model_ready_share,0.110435
model_ready_blocks,349.000000
temporal_gap_rows,68347.000000
gmp_quality_rows,154999.000000
full_3h_lookback_rows,63219.000000
valid_next_hour_target_rows,136718.000000



Model-ready samples per block:


,samples_per_block
count,349.000000
mean,145.988539
std,84.514400
min,2.000000
25%,108.000000
50%,136.000000
75%,182.000000
max,477.000000


In [25]:
# 30. Preview a model-ready sequence

if len(model_ready_df):
    example_block = model_ready_df.iloc[0]["BLOCK_ID"]
    cols = [
        "BLOCK_ID", "START_TIME_DT",
        "GMP_OCCUPANCY", "GMP_AVAILABILITY",
        "GMP_OCC_LAG_1", "GMP_OCC_LAG_2", "GMP_OCC_LAG_3",
        "GMP_ROLLING_MEAN_3",
        "RATE", "RATE_CHANGE",
        "TARGET_GMP_AVAILABILITY_1H", "TARGET_GMP_OCCUPANCY_1H"
    ]
    display(model_ready_df.loc[model_ready_df["BLOCK_ID"].eq(example_block), cols].head(LOOKBACK_HOURS + 3))
else:
    print("No model-ready rows. Inspect the quality and sequence diagnostics.")


,BLOCK_ID,START_TIME_DT,GMP_OCCUPANCY,GMP_AVAILABILITY,GMP_OCC_LAG_1,GMP_OCC_LAG_2,GMP_OCC_LAG_3,GMP_ROLLING_MEAN_3,RATE,RATE_CHANGE,TARGET_GMP_AVAILABILITY_1H,TARGET_GMP_OCCUPANCY_1H
13,902,2011-04-04 13:00:00,0.712599,0.287401,0.781784,0.677296,0.636519,0.723893,2.0,0.0,0.333599,0.666401
14,902,2011-04-04 14:00:00,0.666401,0.333599,0.712599,0.781784,0.677296,0.720261,2.0,0.0,0.419704,0.580296
20,902,2011-04-04 21:00:00,0.453395,0.546605,0.588278,0.488790,0.559623,0.510154,2.0,0.0,0.694302,0.305698
21,902,2011-04-04 22:00:00,0.305698,0.694302,0.453395,0.588278,0.488790,0.449123,2.0,0.0,0.727895,0.272105
75,902,2011-08-04 21:00:00,0.566235,0.433765,0.660623,0.823457,0.869241,0.683438,2.0,0.0,0.579840,0.420160
76,902,2011-08-04 22:00:00,0.420160,0.579840,0.566235,0.660623,0.823457,0.549006,2.0,0.0,0.745426,0.254574


## 31. Export outputs

Three files are written:

- `sfpark_feature_engineered_complete_v2_2_lookback3.csv` — all cleaned observations with engineered features, targets, and diagnostics.
- `sfpark_feature_engineered_model_ready_v2_2_lookback3.csv` — only rows that satisfy the full 3-hour supervised-data requirements.
- `sfpark_feature_schema_v2_2_lookback3.csv` — concise metadata describing the feature roles.


In [26]:
# 32. Export feature-engineered datasets

COMPLETE_OUTPUT = Path("sfpark_feature_engineered_complete_v2_2_lookback3.csv")
MODEL_READY_OUTPUT = Path("sfpark_feature_engineered_model_ready_v2_2_lookback3.csv")
SCHEMA_OUTPUT = Path("sfpark_feature_schema_v2_2_lookback3.csv")

feature_df.to_csv(COMPLETE_OUTPUT, index=False)
model_ready_df.to_csv(MODEL_READY_OUTPUT, index=False)

schema_rows = []
for c in MODEL_INPUT_FEATURES:
    schema_rows.append({"column": c, "role": "model_input", "future_information": False})
for c in TARGET_COLUMNS:
    schema_rows.append({"column": c, "role": "target", "future_information": True})
for c in IDENTIFIER_COLUMNS:
    schema_rows.append({"column": c, "role": "identifier_or_descriptor", "future_information": False})
for c in QUALITY_COLUMNS:
    schema_rows.append({"column": c, "role": "quality_or_sequence_diagnostic", "future_information": False})

pd.DataFrame(schema_rows).to_csv(SCHEMA_OUTPUT, index=False)

print("Saved:")
print(" -", COMPLETE_OUTPUT.resolve())
print(" -", MODEL_READY_OUTPUT.resolve())
print(" -", SCHEMA_OUTPUT.resolve())


Saved:
 - C:\Users\Pulkit Gupta\Downloads\sfpark_feature_engineered_complete_v2_2_lookback3.csv
 - C:\Users\Pulkit Gupta\Downloads\sfpark_feature_engineered_model_ready_v2_2_lookback3.csv
 - C:\Users\Pulkit Gupta\Downloads\sfpark_feature_schema_v2_2_lookback3.csv


## 33. Hand-off to the forecasting notebook

The next notebook should construct LSTM samples using:

- `BLOCK_ID` as spatial identity (preferably a learned embedding rather than numeric magnitude),
- exactly `LOOKBACK_HOURS = 3` chronological rows per sample,
- `TARGET_GMP_AVAILABILITY_1H` as the primary regression target.

Do **not** calculate block-level normalization statistics, target encodings, or block historical baselines on the full dataset before the chronological train/validation/test split. Those quantities must be fit using training data only to prevent temporal leakage.
